# Result height stability

Timestamp: 2026-09-11 23:50:26 +04


## Hypothesis

A fixed `height: 2.5em` at the existing unitless `line-height: 1.25` reserves exactly two text lines. Keeping `min-height: 58px` preserves the existing floor, while `overflow: hidden` clips only text beyond the reserved area.


## Method

Extract the `.result` declaration and assert all required properties. Extract and execute the existing inline JavaScript in Node with a mocked DOM, force deterministic random values, and assert exact wording, 20 choices, and no consecutive duplicate results. Run `node --check` on the extracted script.


In [1]:
from pathlib import Path
import re
html = Path("../index.html").read_text()
block = re.search(r"\.result\s*\{([^}]*)\}", html).group(1)
required = ["height: 2.5em", "min-height: 58px", "overflow: hidden", "line-height: 1.25"]
missing = [item for item in required if item not in block]
assert not missing, missing
print("CSS_CHECK PASS")
print(".result{" + " ".join(line.strip() for line in block.splitlines() if line.strip()) + "}")


CSS_CHECK PASS
.result{height: 2.5em; min-height: 58px; overflow: hidden; margin: 30px 0 0; font-family: Georgia, "Times New Roman", serif; font-size: clamp(1.35rem, 3vw, 2rem); font-weight: 700; line-height: 1.25;}


In [2]:
import subprocess
node_check = r"""const fs=require("fs"),vm=require("vm"),assert=require("assert");
const html=fs.readFileSync("../index.html","utf8");
const script=html.slice(html.indexOf("<script>")+8,html.indexOf("</script>"));
const listeners={};
const makeEl=()=>({textContent:"",src:"",alt:"",hidden:false,offsetWidth:0,
  classList:{values:new Set(),add(x){this.values.add(x)},remove(x){this.values.delete(x)}},
  addEventListener(type,fn){listeners[type]=fn}});
const elements={"#generate-button":makeEl(),"#result":makeEl(),"#food-photo":makeEl(),
  "#photo-placeholder":makeEl(),"#photo-credit":makeEl()};
const mockMath=Object.create(Math); mockMath.random=()=>0;
const context={document:{querySelector:s=>elements[s]},Math:mockMath};
vm.runInNewContext(script+"\nglobalThis.__names=lunchOptions.map(x=>x.name);",context);
assert.strictEqual(context.__names.length,20);
const outputs=[];
for(let i=0;i<40;i++){listeners.click(); outputs.push(elements["#result"].textContent)}
assert.strictEqual(outputs[0],"Today's pick: Pizza! 🥳");
assert(outputs.every((x,i)=>x===`Today's pick: ${context.__names[i%2]}! 🥳`));
assert(outputs.every((x,i)=>i===0||x!==outputs[i-1]));
console.log("JS_MOCK_CHECK PASS");
console.log(`choices=${context.__names.length}`);
console.log(`first=${outputs[0]}`);
console.log(`clicks=${outputs.length} consecutive_duplicates=0`);"""
completed = subprocess.run(["node", "-e", node_check], check=True, text=True, capture_output=True)
print(completed.stdout, end="")


JS_MOCK_CHECK PASS
choices=20
first=Today's pick: Pizza! 🥳
clicks=40 consecutive_duplicates=0


In [3]:
import subprocess
from pathlib import Path
html = Path("../index.html").read_text()
script = html[html.index("<script>") + 8:html.index("</script>")]
subprocess.run(["node", "--check"], input=script, check=True, text=True, capture_output=True)
print("INLINE_JS_SYNTAX PASS")


INLINE_JS_SYNTAX PASS


## Interpretation

The CSS check found the exact two-line block height, retained minimum height and line height, and overflow clipping. The mocked JavaScript check retained the exact result wording, all 20 choices, and zero consecutive duplicates across 40 deterministic clicks. Inline JavaScript syntax also passed.
